# 05 — E-Swap-1..6: Hot-Swap Timeline Analysis

Internal runtime phases, HTTP request duration, and sink-observed output gaps are separate measurements. E-Swap-2 and E-Swap-6 reuse E-Swap-1 leaves; analysis de-duplicates by `measurement_source_leaf` rather than counting copied leaves as new samples.

**Inputs**: canonical `hotswap-analysis.json` leaves under `eval/results/e-swap-{1,2,4,6}/rpi5-<batch>/`.


In [ ]:
import json, os, glob, re
from wafer_analysis.paths import find_latest_shakedown
from wafer_analysis.tables import hotswap_timeline_table
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

REPO = Path.cwd()
while REPO != REPO.parent and not (REPO / 'eval').is_dir():
    REPO = REPO.parent

def find_shakedown(exp_name):
    return find_latest_shakedown(exp_name)

plt.style.use('seaborn-v0_8-paper')
plt.rcParams.update({'figure.dpi': 120, 'font.size': 9})


In [ ]:
# T9: apply thesis-grade matplotlib styling (font.family=serif, pdf.fonttype=42, mathtext=cm).
from wafer_analysis.plots import setup_thesis_style, save_figure
setup_thesis_style()


## Internal phases and sink-observed output gaps

The API-side phases and HTTP total describe swap work. `sink_observed_output_gap_ms` is an output interarrival measurement. Queued output can mask internal disruption by keeping the sink active while an internal swap is in progress, so a smaller sink gap does not establish a faster internal swap.


In [ ]:
frames = []
seen_source_leaves = set()
interpretation = None
for experiment in ('e-swap-1', 'e-swap-2', 'e-swap-4', 'e-swap-6'):
    batch = find_shakedown(experiment)
    for artifact in sorted(batch.glob('**/hotswap-analysis.json')):
        evidence = json.loads(artifact.read_text())
        source_leaf = evidence['measurement_source_leaf']
        if source_leaf in seen_source_leaves:
            continue
        seen_source_leaves.add(source_leaf)
        frame, interpretation = hotswap_timeline_table(evidence)
        frames.append(frame)

assert frames, 'No hotswap-analysis.json artifacts found'
hotswap_df = pd.concat(frames, ignore_index=True)
display(hotswap_df[[
    'experiment', 'condition', 'event_index', 'compile_ms', 'instantiate_ms',
    'signal_ms', 'ack_ms', 'convergence_ms', 'http_total_ms',
    'sink_observed_output_gap_ms', 'measurement_source_leaf',
]])
print(interpretation)
print(f'Unique measurement leaves: {len(seen_source_leaves)}')


## Sink-observed output-gap distribution

This distribution is evaluated against the unchanged RQ3 observed-pause criterion. It is not used as a proxy for compile, instantiate, ACK, convergence, or HTTP duration.


In [ ]:
pause_ms = hotswap_df['sink_observed_output_gap_ms'].to_numpy()
print(f'Sink-observed gap samples: {len(pause_ms)}')
if len(pause_ms) > 0:
    print(f'  p50: {np.percentile(pause_ms, 50):.3f} ms')
    print(f'  p95: {np.percentile(pause_ms, 95):.3f} ms')
    print(f'  p99: {np.percentile(pause_ms, 99):.3f} ms')

fig, ax = plt.subplots(figsize=(8, 4))
if len(pause_ms) > 0:
    ax.hist(pause_ms, bins=30, edgecolor='black', alpha=0.75)
    ax.axvline(100, color='red', linestyle='--', label='Observed-pause target: 100 ms')
ax.set_xlabel('Sink-observed output gap (ms)')
ax.set_ylabel('Count')
ax.set_title('Hot-Swap Sink-Observed Output Gaps')
ax.legend()
plt.tight_layout()
plt.show()
_saved = save_figure(fig, 'e-swap/hotswap_timeline')
print(f'saved thesis figure to {_saved}')


## E-Swap-2: Sequence Integrity Summary

In [ ]:
sd2 = find_shakedown('e-swap-2')
assert sd2 is not None, 'E-Swap-2 shakedown not found'

with open(sd2 / 'shakedown.json') as f:
    e2_data = json.load(f)

# Summary table
summary = pd.DataFrame([{
    'Metric': 'Total Messages Expected',
    'Value': e2_data['total_expected'],
}, {
    'Metric': 'Total Received (post-warmup)',
    'Value': e2_data['total_received'],
}, {
    'Metric': 'Warmup Excluded',
    'Value': e2_data['warmup_excluded_msgs'],
}, {
    'Metric': 'Swap-Induced Gaps',
    'Value': e2_data['swap_induced_gaps'],
}, {
    'Metric': 'Sequence Duplicates',
    'Value': e2_data['sequence_duplicates'],
}, {
    'Metric': 'Total Swaps',
    'Value': e2_data['total_swaps'],
}, {
    'Metric': 'Zero-Loss Invariant',
    'Value': '✓ PASS' if e2_data['zero_loss'] else '✗ FAIL',
}, {
    'Metric': 'Zero-Dup Invariant',
    'Value': '✓ PASS' if e2_data['zero_dup'] else '✗ FAIL',
}])

print(summary.to_string(index=False))

## E-Swap-4: burst-load internal and output observations

Both axes remain visible. Queue masking is reported as observed behavior and is not corrected away.


In [ ]:
burst = hotswap_df[hotswap_df['experiment'] == 'e-swap-4'].copy()
assert not burst.empty, 'E-Swap-4 hotswap-analysis.json not found'
display(burst[[
    'event_index', 'compile_ms', 'instantiate_ms', 'ack_ms', 'convergence_ms',
    'http_total_ms', 'sink_observed_output_gap_ms', 'measurement_source_leaf',
]])

fig, ax = plt.subplots(figsize=(8, 4))
ax.scatter(burst['http_total_ms'], burst['sink_observed_output_gap_ms'], alpha=0.75)
ax.set_xlabel('HTTP total duration (ms)')
ax.set_ylabel('Sink-observed output gap (ms)')
ax.set_title('E-Swap-4: Internal Request Time vs Observed Output Gap')
plt.tight_layout()
plt.show()
print(interpretation)


## E-Swap-5: Failed Swap / Rollback Timeline

In [ ]:
sd5 = find_shakedown('e-swap-5')
assert sd5 is not None, 'E-Swap-5 shakedown not found'

with open(sd5 / 'shakedown.json') as f:
    e5_data = json.load(f)

# Summary
print('E-Swap-5: Failed Swap Recovery')
print(f'  Swap attempts: {e5_data["swap_attempts"]}')
print(f'  Trap count post-swap: {e5_data["trap_count_post_swap"]}')
print(f'  A4 init-rollback events: {e5_data["a4_init_rollback_events"]}')
print(f'  A17 process-time rollback events: {e5_data.get("a17_process_time_rollback_events", 0)}')
print(f'  Auto rollback to v1: {e5_data["auto_rollback_to_v1"]}')
print(f'  Sequence continues after rollback: {e5_data["sequence_continues_after_rollback"]}')
print(f'  Runtime panic: {e5_data["runtime_panic"]}')
print(f'  Messages received: {e5_data["total_received"]}')
print(f'  Sequence gaps: {e5_data["sequence_gaps"]}')

# A17 (Closed 2026-08-02): show per-event rollback times; AC threshold <10s.
rt_ns = e5_data.get('rollback_times_ns') or []
if rt_ns:
    import statistics as _s
    rt_ms = [ns / 1_000_000.0 for ns in rt_ns]
    p50 = _s.median(rt_ms)
    p99 = _s.quantiles(rt_ms, n=100)[98] if len(rt_ms) > 1 else rt_ms[0]
    print()
    print(f'  A17 rollback time (n={len(rt_ns)}): p50={p50:.3f} ms, p99={p99:.3f} ms, max={max(rt_ms):.3f} ms')
    assert max(rt_ms) < 10_000.0, f'AC violated: rollback max {max(rt_ms):.1f} ms >= 10 s'
    print(f'  AC: rollback under 10 s wall-clock -> PASS')
else:
    print()
    print('  (no A17 rollback_time_ns samples recorded)')
print()
print(f'Note: {e5_data["note"]}')

# Timeline visualization: messages received over time
thr_path = sd5 / 'run-1' / 'throughput.csv'
if thr_path.exists():
    thr_df = pd.read_csv(thr_path)
    fig, ax = plt.subplots(figsize=(10, 4))
    ax.plot(thr_df.iloc[:, 0], thr_df.iloc[:, 1], 'b-', linewidth=1.5)
    ax.axvline(6, color='red', linestyle='--', label='First swap attempt (v2-panics)')
    ax.set_xlabel('Elapsed (s)')
    ax.set_ylabel('Messages / bucket')
    ax.set_title('E-Swap-5: Throughput After Failed Swap (A17 canary rollback)')
    ax.legend()
    plt.tight_layout()
    plt.show()
else:
    print('  (throughput.csv not available for visualization)')

# (figure saved in E-Swap-1 cell above)

## De-duplicated measurement-source summary

In [ ]:
summary = (
    hotswap_df.groupby(['measurement_source_leaf', 'experiment', 'condition'], as_index=False)
    .agg(
        swaps=('event_index', 'count'),
        median_http_total_ms=('http_total_ms', 'median'),
        median_sink_observed_output_gap_ms=('sink_observed_output_gap_ms', 'median'),
    )
)
display(summary)
print('Copied E-Swap-2/E-Swap-6 views retain provenance but do not add sample rows.')
print(interpretation)
